# JAX primer

**Artifact `jax-primer`. Session U0.L5, part 2 of 2.** Part 1 is
`lab-u0l5.ipynb`, and section 6 below uses the ODE you solved there.

Three later sessions of this course have a JAX moment in them (U1.L1, U2.L1,
U3.L1). A language switch in the middle of a hard topic is expensive, so the
switch happens here, on a problem you already understand completely, where the
only new thing is the language.

### Why JAX at all

Not fashion. Three reasons, and they are worth stating because the choice
deserves an argument.

1. `grad`, `vmap` and `jit` make several of this course's constructions
   **clearer** than their PyTorch equivalents. The per-sample Jacobians behind
   a coupling layer's log-determinant in U1.L1 are the standard example.
2. **diffrax** is the strongest differential-equation library in the ecosystem,
   and U2.L1 uses it as a professional mirror of the module you wrote in part 1.
3. A large part of this literature is now published as JAX code, and you have to
   be able to read it.

**PyTorch remains this course's primary language.** The JAX moments are always
mirrors, never the only path.

### Scope, declared

Five moves and then a capstone. There is no flax and no optax here beyond this
sentence saying they exist and appear in U3.L1; there is nothing about sharding
or TPUs. This primer optimizes for *reading* research JAX. It does not optimize
for writing production JAX, which is a different skill with a different book.

### The spine

> **JAX is NumPy, plus function transformations, plus explicit randomness.**

Everything below is an elaboration of that sentence.

### Install (CPU, which is all this needs)

```
pip install "jax[cpu]" diffrax
```

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

# Float64 is OFF by default in JAX: every array is float32 unless you say
# otherwise, and the gradient ritual below is meaningless in float32 (U0.L1
# measured it wrong by a factor of eight on a correct gradient). This course
# turns it on explicitly, here, BEFORE any array is created -- never relying
# on JAX's default dtype policy. If your JAX build cannot enable x64, the
# float64 claims of this notebook do not hold and its checks must be read
# at float32 tolerances.
jax.config.update("jax_enable_x64", True)
assert jnp.zeros(1).dtype == jnp.float64, "x64 is not enabled; see the comment"

print("jax    ", jax.__version__)
print("devices", jax.devices())

## 1 — Pure functions and `jnp`

`jnp` is NumPy's API over **immutable** arrays. `x[i] = v` is an error. The
replacement is a functional update that returns a new array and leaves the old
one alone.

In [ ]:
x = jnp.arange(5.0)
y = x.at[2].set(99.0)      # returns a NEW array
print("x:", x)             # unchanged
print("y:", y)

try:
    x[2] = 99.0
except TypeError as e:
    print("\nin-place assignment:", type(e).__name__)
    print(str(e).splitlines()[0])

### The price, honestly

You give up in-place mutation, and with it a family of comfortable patterns.

What you buy is that a JAX function can be **transformed** — differentiated,
vectorized, compiled, or all three at once. A transformation needs to know that
calling your function twice with the same arguments does the same thing twice.
A function that reads a global counter, appends to a list, or prints, cannot be
safely traced.

**Purity is the price of transformability.** The four moves below are what it
purchases.

### TODO 1

Write a pure function that normalises a vector to zero mean and unit variance,
using no in-place operations.

In [ ]:
def standardize(v):
    """Return (v - mean) / std. Pure: no mutation of v."""
    # TODO 1
    raise NotImplementedError

In [ ]:
v = jnp.array([1.0, 2.0, 3.0, 4.0])
out = standardize(v)
assert jnp.allclose(jnp.mean(out), 0.0, atol=1e-12)
assert jnp.allclose(jnp.std(out), 1.0, atol=1e-12)
assert jnp.allclose(v, jnp.array([1.0, 2.0, 3.0, 4.0])), "v was mutated"
print("passed — and v is untouched:", v)

## 2 — `grad`

`jax.grad(f)` returns a **function**: the gradient of `f` with respect to its
first argument. `f` must return a scalar.

There is no tape, no `.backward()`, and no `.grad` attribute on anything. The
gradient is a function of the same shape as the original.

In [ ]:
def loss(theta, x):
    return jnp.sum((theta * x - 1.0) ** 2)

dloss = jax.grad(loss)                 # same signature as loss
theta = jnp.array([0.5, -1.5, 2.0])
xs = jnp.array([1.0, 2.0, 0.5])

print("loss    ", loss(theta, xs))
print("grad    ", dloss(theta, xs))

# Both outputs for the price of one — you will want this in every training loop.
value, grad = jax.value_and_grad(loss)(theta, xs)
print("together", value, grad)

### TODO 2 — the directional finite-difference ritual, third appearance

U0.L1 introduced it. U0.L2 repeated it. Here it is a third time, against
`jax.grad`. The ritual is **directional**: draw one fixed direction $v$ with
$\lVert v \rVert = 1$, and compare the directional derivative autograd reports
with the central difference along that same direction,

$$
\big\langle \nabla f(\theta),\, v \big\rangle
\approx
\frac{f(\theta + \varepsilon v) - f(\theta - \varepsilon v)}{2\varepsilon},
\qquad \text{error } O(\varepsilon^2).
$$

A direction and not a coordinate, for the reason U0.L1 gave: the right-hand
side costs **two evaluations whatever the parameter count**, so the same
ritual still works on the vector field of U3.L1. The three conditions of
U0.L1 travel with it: run in float64 (this notebook enabled x64 in its first
cell), freeze every random draw across the three evaluations (this loss has
none), and accept on a mixed tolerance.

By now this should be a reflex. It costs three lines, it takes a second, and it
is the only thing that distinguishes a gradient from a plausible array of
numbers.

In [ ]:
def directional_fd(f, theta, v, x, eps=1e-6):
    """Central difference of f along the unit direction v."""
    # TODO 2: two evaluations of f, one subtraction. No in-place writes.
    raise NotImplementedError

In [ ]:
key = jax.random.key(0)                      # the draw is fixed, and visible
v = jax.random.normal(key, theta.shape)
v = v / jnp.linalg.norm(v)

d_fd = float(directional_fd(loss, theta, v, xs))
d_ad = float(jnp.dot(dloss(theta, xs), v))    # autograd's side: <grad, v>
print("finite difference along v:", d_fd)
print("<jax.grad, v>            :", d_ad)
print("deviation                :", abs(d_fd - d_ad))
assert abs(d_fd - d_ad) <= 1e-8 + 1e-6 * abs(d_ad), "the gradient is wrong"
print("\nRitual passed.")

# The same number a third way, and a JAX-only one: jax.jvp IS the forward-mode
# pass along v. This is the cost table of U0.T1 made executable.
_, d_jvp = jax.jvp(lambda th: loss(th, xs), (theta,), (v,))
assert abs(float(d_jvp) - d_ad) < 1e-12

## 3 — `vmap`

`jax.vmap` maps a function over a batch axis **without the function knowing
about batches**. You write the mathematics for one sample; `vmap` supplies the
axis.

For a forward pass that is a convenience. For a gradient it is considerably
more, because the quantity below is genuinely awkward to obtain in eager
PyTorch — *eager* meaning the ordinary mode in which each operation runs as the
interpreter reaches it — and the awkwardness is why people never look at it.

In [ ]:
key = jax.random.key(0)
key, k_w1, k_w2, k_x, k_y = jax.random.split(key, 5)

d_in, d_hidden, batch = 8, 32, 512
params = {
    "w1": jax.random.normal(k_w1, (d_in, d_hidden)) / jnp.sqrt(d_in),
    "b1": jnp.zeros(d_hidden),
    "w2": jax.random.normal(k_w2, (d_hidden, 1)) / jnp.sqrt(d_hidden),
    "b2": jnp.zeros(1),
}
xs_b = jax.random.normal(k_x, (batch, d_in))
ys_b = jax.random.normal(k_y, (batch, 1))


def loss_one(params, x, y):
    """Squared error on ONE sample. No batch dimension in the body."""
    h = jnp.tanh(x @ params["w1"] + params["b1"])
    pred = h @ params["w2"] + params["b2"]
    return jnp.sum((pred - y) ** 2)


print("one sample's loss:", loss_one(params, xs_b[0], ys_b[0]))

### TODO 3 — per-sample gradients

Compose `vmap` and `grad` to get **one gradient per sample**.

`in_axes` says which arguments are mapped: `None` for the shared parameters,
`0` for the inputs and the targets.

In [ ]:
# TODO 3: per_sample should be a pytree with the same structure as `params`,
# each leaf carrying a leading axis of length `batch`.
per_sample = ...
raise NotImplementedError

### Check it before you believe it

`vmap(grad(f))` on one sample, against `grad(f)` on that sample in a plain
Python loop. The deviation should be **zero to machine precision**: `vmap` is
not an approximation of the loop, it is the same computation with a different
execution order.

In [ ]:
loop_g0 = jax.grad(loss_one)(params, xs_b[0], ys_b[0])
max_dev = max(float(jnp.max(jnp.abs(loop_g0[k] - per_sample[k][0])))
              for k in loop_g0)
print("max deviation, vmap against loop:", max_dev)
assert max_dev < 1e-10
print("shapes:", {k: per_sample[k].shape for k in per_sample})

### What the per-sample view shows

Now look at the distribution the batch gradient hides.

In [ ]:
import matplotlib.pyplot as plt

norms = np.array([
    float(jnp.sqrt(sum(jnp.sum(per_sample[k][i] ** 2) for k in per_sample)))
    for i in range(batch)
])
mean_grad = jax.tree.map(lambda g: g.mean(axis=0), per_sample)
norm_of_mean = float(jnp.sqrt(sum(jnp.sum(l ** 2)
                                  for l in jax.tree.leaves(mean_grad))))

print(f"mean of the norms : {norms.mean():.3f}")
print(f"norm of the mean  : {norm_of_mean:.3f}")
print(f"ratio             : {norms.mean() / norm_of_mean:.1f}x")
print(f"spread            : {norms.min():.4f} to {norms.max():.2f}")

fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.hist(norms, bins=48, color="#00447f", edgecolor="white", lw=0.4)
ax.axvline(norms.mean(), color="#6b7280", ls=":", lw=1.6,
           label=f"mean of the norms = {norms.mean():.2f}")
ax.axvline(norm_of_mean, color="#d97a1f", lw=2.0,
           label=f"norm of the mean = {norm_of_mean:.2f}")
ax.set_xlabel("gradient norm"); ax.set_ylabel("samples")
ax.legend(frameon=False); ax.grid(axis="y", color="#DDDDDD", lw=0.6)
plt.tight_layout(); plt.show()

The gap between the two lines is **cancellation**: the per-sample gradients
point in partly opposing directions, so their average is shorter than its parts.
That is not a pathology — it is what averaging a batch *is*. But it means the
number the optimizer sees is not a typical sample's number, and the spread
behind it is invisible from the batch gradient alone.

**Where this returns.** U1.L1 computes the log-determinant of a coupling
layer's Jacobian, per sample, with `jax.jacfwd`. Same composition, different
inner transformation. This TODO is its warm-up.

## 4 — `jit`

`jax.jit(f)` compiles `f` with XLA. The whole tracing model you need today fits
in three sentences.

1. JAX runs your Python **once**, with abstract placeholder values, and records
   the operations. That recording is what gets compiled.
2. The recording is cached per input **shape and dtype**, so a new shape
   re-traces and re-compiles.
3. Because the trace sees placeholders rather than numbers, Python control flow
   that branches on a **value** cannot be recorded.

### The trap, planted rather than described

Run the next cell. Read the error. An error message you have met once is a
different thing from an error message you have read about — and this is the one
every newcomer to JAX meets first.

In [ ]:
@jax.jit
def bad_abs(x):
    if x > 0:        # branches on a VALUE — the tracer has none
        return x
    return -x

try:
    bad_abs(3.0)
except Exception as e:
    print(type(e).__name__)
    print(str(e).splitlines()[0])

### TODO 4 — fix it

`jnp.where(cond, a, b)` selects elementwise and computes both branches, which
is exactly what a traced program can do.

In [ ]:
@jax.jit
def good_abs(x):
    # TODO 4: same result, no Python branch on a value.
    raise NotImplementedError

In [ ]:
for v in (3.0, -3.0, 0.5):
    assert float(good_abs(v)) == abs(v)
print("passed:", [float(good_abs(v)) for v in (3.0, -3.0, 0.5)])

# Sentence 2, made visible: a new SHAPE re-traces.
traced = []


@jax.jit
def f(x):
    traced.append(x.shape)     # runs at TRACE time, not at call time
    return jnp.sum(x ** 2)


for shape in [(3,), (3,), (4,), (4,), (3,)]:
    f(jnp.ones(shape))
print("\ntraces recorded:", traced)
print(f"five calls, two distinct shapes -> {len(traced)} traces")
print("the fifth call reused the first trace, because (3,) was already seen")

## 5 — Pytrees and PRNG keys

A **pytree** is any nest of containers with arrays at the leaves. A dict of
dicts of arrays is the usual shape of a parameter set, and it is what `params`
above already is.

Every JAX transformation accepts and returns pytrees. That is why `jax.grad` of
a loss over a dict of parameters returns a dict of the same shape, and why
`jax.tree.map` applies an update elementwise across the whole nest.

In [ ]:
print("structure:", jax.tree.structure(params))
print("leaf shapes:", [leaf.shape for leaf in jax.tree.leaves(params)])

# One gradient-descent step, as a tree operation. This is the whole of an
# optimizer without a library.
lr = 0.01
updated = jax.tree.map(lambda p, g: p - lr * g, params, mean_grad)
print("\nsame structure after the update:",
      jax.tree.structure(updated) == jax.tree.structure(params))

### Explicit randomness

There is no global seed and no hidden state. A random function takes a **key**
and returns the same value for the same key, always. To get fresh randomness you
**split** the key, and you use each key once.

In [ ]:
key = jax.random.key(0)

a = jax.random.normal(key, (3,))
b = jax.random.normal(key, (3,))
print("same key twice   :", a, b)
print("identical?       :", bool(jnp.all(a == b)))

key, sub1 = jax.random.split(key)
key, sub2 = jax.random.split(key)
print("\nafter two splits :", jax.random.normal(sub1, (3,)),
      jax.random.normal(sub2, (3,)))

### TODO 5 — a seeded pipeline with no global state

Write a function that draws a batch of noise and a batch of labels from **one**
incoming key, and returns the updated key alongside them. This is the pattern
every JAX training loop uses.

The rule to respect: **never use a key twice.**

In [ ]:
def draw_batch(key, n, d):
    """Return (new_key, noise (n, d), labels (n,)). Never reuse a key."""
    # TODO 5
    raise NotImplementedError

In [ ]:
k = jax.random.key(42)
k, noise1, lab1 = draw_batch(k, 4, 3)
k, noise2, lab2 = draw_batch(k, 4, 3)
assert not bool(jnp.all(noise1 == noise2)), "the key was reused"

# Reproducibility: the same starting key gives the same whole sequence.
k2 = jax.random.key(42)
k2, n1, l1 = draw_batch(k2, 4, 3)
assert bool(jnp.all(n1 == noise1)) and bool(jnp.all(l1 == lab1))
print("fresh draws differ; the same seed reproduces exactly.")

### Why this section is in a course about generative models

U0.L2 made reproducibility a **discipline**: seed the three generators, record
the seed, remember not to call a random function in a place that shifts the
stream.

Here it is not a discipline. A key you forgot to split is a bug **you can see in
the code**, and a function's randomness is visible in its signature.

**JAX makes the seeding discipline structural. Randomness is an argument;
reproducibility is the default.**

## 6 — Capstone: the same ODE, in diffrax

The problem of part 1, solved again by somebody else's solver.

diffrax wants `f(t, y, args)`. You have `f(t, x)`. The adapter is two lines, and
writing it teaches the shape of the API.

In [ ]:
import diffrax

print("diffrax", diffrax.__version__)

T_END = 2 * np.pi
Y0 = jnp.array([1.0, 0.0])

### TODO 6 — the adapter and the solve

`diffrax.ODETerm` wraps the right-hand side; `diffrax.Tsit5()` is the tableau;
`diffrax.PIDController(rtol=..., atol=...)` is the step-size rule.

`Tsit5` is a Runge–Kutta tableau of order 5 with an embedded estimate of order 4
— the same construction as your adaptive Heun, with better coefficients and more
stages. `dt0` is the first step, which the controller then overrides.

**NFE is instrumented, not inferred.** diffrax reports `num_steps`,
`num_accepted_steps` and `num_rejected_steps` in `sol.stats`, and no count of
evaluations. "Six per step" is a fact about the tableau you would have to take
on trust. So the field below counts its own calls with `jax.debug.callback`,
which fires once per real evaluation, under `jit` and inside the solver's loop.
Read all three step counts too; they are what the library actually exposes.

In [ ]:
CALLS = {"n": 0}


def _count_call():
    CALLS["n"] += 1

In [ ]:
def harmonic_jax(t, y, args):
    """dx/dt = v, dv/dt = -x, in diffrax's signature."""
    # TODO 6a: the field -- with jax.debug.callback(_count_call) as its
    # first line, so that NFE is counted rather than inferred.
    raise NotImplementedError


def solve(rtol, atol):
    """Integrate to T_END with Tsit5. Return (y1, nfe, stats)."""
    # TODO 6b
    raise NotImplementedError

### The cross-check

Two independent solvers, one problem with a known answer: the exact orbit
returns to $(1, 0)$ at $t = 2\pi$. So each solver is checked against the
**truth** first, and only then against the other. Two solvers that agree with
each other and not with the truth would be two wrong solvers, and this check
would catch it.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "labs" else Path.cwd()
sys.path.insert(0, str(REPO))

import torch
from common.odesolve import odeint

torch.set_default_dtype(torch.float64)


def harmonic_torch(t, x):
    return torch.stack([x[:, 1], -x[:, 0]], dim=1)


EXACT = np.array([1.0, 0.0])

y_dfx, nfe_dfx, stats = solve(1e-12, 1e-14)
x_ours = odeint(harmonic_torch, torch.tensor([[1.0, 0.0]]), 0.0, float(T_END),
                method="rk4", n_steps=4000)

err_dfx = float(np.linalg.norm(np.asarray(y_dfx) - EXACT))
err_ours = float(np.linalg.norm(x_ours.numpy()[0] - EXACT))
diff = float(np.linalg.norm(np.asarray(y_dfx) - x_ours.numpy()[0]))
print(f"diffrax Tsit5 (rtol 1e-12): error vs exact {err_dfx:.2e}  "
      f"[NFE {nfe_dfx}; {stats}]")
print(f"your RK4 (4000 steps)    : error vs exact {err_ours:.2e}  "
      f"[NFE {x_ours.nfe}]")
print(f"\ndifference between them  : {diff:.2e}")
assert err_dfx < 1e-9, "diffrax is off the exact solution at a tight tolerance"
assert err_ours < 1e-9, "your RK4 is off the exact solution at a fine step"
assert diff < 1e-9, "one of the two solvers is wrong"
print("Both solvers are within 1e-9 of the truth. Now the rest of the news.")

### The rest of the news

Correct is not the same as competitive. The table runs both solvers at the
same `rtol` values — an **API demonstration**, because the two controllers
read a tolerance differently and the same request does not buy the same
accuracy. The comparison that means something is by **columns**: read the
error each achieved, and the NFE it cost.

In [ ]:
print(f"{'rtol':>8s} {'ours NFE':>10s} {'ours err':>11s} "
      f"{'Tsit5 NFE':>11s} {'Tsit5 err':>11s}")
for rtol in (1e-4, 1e-6, 1e-8):
    x1 = odeint(harmonic_torch, torch.tensor([[1.0, 0.0]]), 0.0, float(T_END),
                method="adaptive_heun", rtol=rtol, atol=rtol / 100)
    e_ours = float((x1 - torch.tensor([[1.0, 0.0]])).norm())
    y1, nfe_t5, _ = solve(rtol, rtol / 100)
    e_dfx = float(np.linalg.norm(np.asarray(y1) - np.array([1.0, 0.0])))
    print(f"{rtol:>8.0e} {int(x1.nfe):>10d} {e_ours:>11.2e} "
          f"{nfe_t5:>11d} {e_dfx:>11.2e}")

Read that table twice.

**Humility first.** Your solver is correct — the cross-check proved it — and it
costs orders of magnitude more than the tool a specialist wrote. The gap widens
as the tolerance tightens, because an order-5 cost curve is much flatter than an
order-2 one.

**Then continuity.** Writing your own was still necessary: you now know what
`Tsit5`, `rtol` and `PIDController` mean, and you know it because you built the
smaller version of each. The from-scratch pass is how you learn the object. It
is not how you integrate an equation you care about. U2.L1 uses diffrax and
`torchdiffeq` as the professional mirrors of your module, and re-explains none
of their arguments.

### Honest timing

The last measurement, and the rule outlives the number.

> **Any claim in this course about the speed of a JAX function reports compile
> time separately from steady-state time, and never amortises the compile into
> the speedup.**

The rule exists because the alternative is easy and wrong in both directions.
Time the first call and `jit` looks several times *slower* than eager. Time only
the steady state and quote it as the speedup, and you have hidden a fixed cost
that a short-running script never earns back.

In [ ]:
import time


def work(a):
    h = a
    for _ in range(12):
        h = jnp.tanh(h @ (h.T @ h) / a.shape[0])
    return jnp.sum(h ** 2)


key = jax.random.key(1)
a = jax.random.normal(key, (4096, 64))


def median_time(fn, arg, repeats):
    # The MEDIAN, not the mean: one scheduler hiccup moves a mean.
    samples = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        jax.block_until_ready(fn(arg))
        samples.append(time.perf_counter() - t0)
    return float(np.median(samples))


jax.block_until_ready(work(a))          # warm the eager path too — fair is fair
eager = median_time(work, a, 15)

jitted = jax.jit(work)
t0 = time.perf_counter()
jax.block_until_ready(jitted(a))
first = time.perf_counter() - t0

steady = median_time(jitted, a, 41)
compile_only = first - steady

print(f"eager            : {eager*1e3:7.1f} ms")
print(f"first jitted call: {first*1e3:7.1f} ms  "
      f"(of which ~{compile_only*1e3:.0f} ms is compilation)")
print(f"steady state     : {steady*1e3:7.1f} ms")
print(f"\nspeedup     : {eager/steady:.1f}x per call")
print(f"break-even  : after about {compile_only/(eager-steady):.0f} calls")

### Two more honesties

**A modest speedup is the real one.** This workload is a chain of matrix
multiplications, and a matrix multiplication is already a call into a tuned
kernel with little for a compiler to improve. XLA earns its large factors by
*fusing* many small operations, and there are few here. Reporting the factor you
measured rather than the one you hoped for is the entire point of measuring.

**Wall-clock time does not reproduce.** Every other number in this notebook
comes back identical on a re-run. These three do not: they move between runs on
an unloaded machine, and they will be different on your machine. What reproduces
is the shape — compilation dominates the first call, steady state beats eager,
break-even is on the order of ten calls.

Quote a timing with the machine it was measured on, or do not quote it.

## What you can now read

| Move | What it gave you | Where it returns |
|---|---|---|
| pure functions, `jnp` | transformable code | everywhere below |
| `grad` | gradients without a tape | U3.L1 |
| `vmap` | per-sample quantities | **U1.L1** — per-sample log-dets |
| `jit` | compilation, and its cost | every timing claim |
| pytrees, PRNG keys | structural reproducibility | U2.L1, U3.L1 |
| diffrax | a professional ODE solver | **U2.L1** |

That is the whole primer. It is deliberately small: it is enough to read the JAX
in this course's papers and to follow this course's JAX moments, and it stops
there.